# Introduction

In this tutorial, we will use the **UNI model** from [MahmoodLab](https://github.com/mahmoodlab/UNI) (follow the Installation steps), a self-supervised vision transformer trained on histology slides, to extract **patch-level feature embeddings** and **token-level feature embeddings** from Whole Slide Images (WSIs).

The steps we will follow are:

1. **Load** the pretrained UNI model and its checkpoint weights.
2. **Tokenize** the WSI into small, non-overlapping patches of size **224×224 pixels**.
3. **Save** each patch image and record its spatial location (X, Y coordinates) on the original WSI.
4. **Feed** the patches through the pretrained UNI model to extract deep **feature embeddings**.
5. **Link** each embedding with its spatial metadata (Slide ID, Patch ID, coordinates).
6. **Save** the final dataset for downstream spatial analysis tasks.

---

By following this process, we create a compact and meaningful representation of very large WSIs, which can be easily used for downstream tasks.


In [ ]:
import anndata as ad
import gc
import glob
import math
import numpy as np
import os
import pandas as pd
from pathlib import Path
from PIL import Image
import timm
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loading the Pretrained UNI Model and Preparing Image Transformations

In this section, we load the **pretrained UNI model** and prepare the **transformations** that will be applied to the image patches before feeding them into the model.

Steps:

1. **Import additional libraries**:
   - `timm`: a popular library that provides implementations of vision transformer models.
   - `huggingface_hub`: for downloading model weights (optional here if already downloaded).

2. **Set the local directory**:
   - `local_dir` is where the pretrained model checkpoint (`pytorch_model.bin`) is stored. Here 'checkpoint_path' is an exampl, set where your checkpoint is located.
   - (Optional) If not downloaded yet, you could use `hf_hub_download` to fetch the weights from Hugging Face.

3. **Model setup**:
   - We define the model architecture using `timm_kwargs` to match the configuration used by UNI (based on a Vision Transformer (ViT) backbone).
   - The model is created with `timm.create_model(**timm_kwargs)`.
   - Pretrained weights are loaded with `model.load_state_dict(...)`.
   - The model is set to evaluation mode (`model.eval()`) and moved to the selected device (CPU or GPU).

4. **Image preprocessing**:
   - Before passing patches to the model, we apply a series of transformations:
     - **Resize** the patch to 224×224 pixels.
     - **Convert** the image to a PyTorch tensor.
     - **Normalize** pixel values using ImageNet statistics (mean and std).



In [ ]:
# weights download: hf_hub_download("MahmoodLab/UNI2-h", filename="pytorch_model.bin", local_dir=weights_dir)
weights_dir = '../weights/'

timm_kwargs = {'model_name': 'vit_giant_patch14_224',
    'img_size': 224, 
               'patch_size': 14, 
               'depth': 24,
               'num_heads': 24,
               'init_values': 1e-5, 
               'embed_dim': 1536,
               'mlp_ratio': 2.66667*2,
               'num_classes': 0, 
               'no_embed_class': True,
               'mlp_layer': timm.layers.SwiGLUPacked, 
               'act_layer': torch.nn.SiLU, 
               'reg_tokens': 8, 
               'dynamic_img_size': True
              }
model = timm.create_model(**timm_kwargs)
model.load_state_dict(torch.load(os.path.join(weights_dir, "pytorch_model.bin"), map_location="cpu"), strict=True)
model.eval()
model.to(device)
transform = transforms.Compose(
    [
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)


# Tokenizing Whole Slide Images (WSIs) into Patches

In this section, we split the WSI of each sample into non-overlapping patches, save them as `.png` files and record their spatial metadata.

**Variables**
- `bin_size`: Visium HD bin size (002, 008 or 016 µm).
- `patch_size` / `stride`: `224 × 224` pixels / `224` pixels (non-overlapping).
- `skip_samples`: sample folders that are never processed.
- `rerun_full`: `False` only processes samples that dont have patches and patch metadata yet. 
`True` processes ALL samples that are not skipped and **deletes** their existing patches and patch metadata first (embedding files in the metadata folder are not touched).

**Conventions**  
Filenames/filepaths start with `f_`, directories (other than the base directory) with `dir_`. If a datatype is needed, it is added at the end (e.g. `f_metadata_csv`). No path has to be edited if you follow the same folder structure.

**Files and paths**
- `f_metadata_csv` (`WSI_patch_metadata.csv`): created here, stores patch names, positions and paths.
- `f_metadata_h5ad` (`WSI_patch_metadata.h5ad`): created during embedding creation, stores patch IDs and embeddings.
- `basedir`: base data directory. `dir_combined`: metadata directory of the `combined` folder, which contains information for ALL samples.
- Per sample (every folder in `basedir` except `combined` and `skip_samples`): `dir_sample`, `dir_wsi` (the `.btf` file in `input`, or `HE.tif` if there is none), `dir_base_output` (transcriptomics information), `dir_metadata` and `dir_patches`.

**Processing**
1. Read the pixel positions of all bins from `tissue_positions.parquet` (spaceranger output) and take the top left and bottom right corner. This crops only the needed frame from the very large WSI file.
2. Load the WSI with `pyvips`, crop the frame and convert it to a PIL image.
3. Crop patches at regular intervals and save each as `.png`.
4. Record Slide ID, patch filename, top-left (X, Y) position in the WSI and patch path.

**Output and errors**  
The metadata is saved to `WSI_patch_metadata.csv` in the sample's metadata folder and in the combined metadata folder (old rows of that sample are replaced). If a sample fails, it is added to `failed_samples`, no metadata is written for it and the other samples are still processed.

The metadata is also needed to link the embeddings back to their location on the WSI.


In the following, please choose the bin size (2um, 8um or 16um), the `skip_samples` and whether to rerun everything (`rerun_full`). All sample folders in the data directory are checked automatically.

In [ ]:
# Basic Variables
bin_size = "002"  # Adapt as needed

patch_size = (224, 224) # determines patch size in pixels 
stride = 224

skip_samples = ["new", "CRC_P1", "CRC_P2", "CRC_P5", "Normal_P3", "Normal_P5"]  # folders that are never processed
rerun_full = False  # True: delete patches + patch metadata of ALL non-skipped samples and recreate them; False: only samples without patches/metadata


# Filenames
f_metadata_csv = "WSI_patch_metadata.csv"   
f_metadata_h5ad = "WSI_patch_metadata.h5ad"  


# Paths
basedir = Path("../../../../data") # base directory containing all data

dir_combined = basedir / "combined/01_preprocessing/unimodel/metadata"
dir_combined.mkdir(parents=True, exist_ok=True)


def get_dir_patches(dir_sample: Path) -> Path:
    return dir_sample / "01_preprocessing/unimodel/patches"

def get_dir_metadata(dir_sample: Path) -> Path:
    return dir_sample / "01_preprocessing/unimodel/metadata"

def has_patches_and_metadata(dir_sample: Path) -> bool:
    dir_patches = get_dir_patches(dir_sample)
    has_patches = dir_patches.is_dir() and any(dir_patches.glob("*.png"))
    return has_patches and (get_dir_metadata(dir_sample) / f_metadata_csv).exists()

def find_wsi(dir_sample: Path) -> Path:
    # Visium HD samples: single .btf in input/, SMAP samples: HE.tif
    btf_files = sorted((dir_sample / "input").glob("*.btf"))
    if len(btf_files) == 1:
        return btf_files[0]
    if len(btf_files) == 0 and (dir_sample / "input/HE.tif").exists():
        return dir_sample / "input/HE.tif"
    raise FileNotFoundError(f"Could not determine WSI in {dir_sample / 'input'} (found: {[f.name for f in btf_files]})")


# Select samples
all_samples = sorted(
    p.name for p in basedir.iterdir()
    if p.is_dir() and p.name != "combined" and p.name not in skip_samples
)

if rerun_full:
    samples_to_process = all_samples
else:
    samples_to_process = [s for s in all_samples if not has_patches_and_metadata(basedir / s)]

print(f"Found {len(all_samples)} samples (excluding 'combined' and skip_samples): {all_samples}")
print(f"rerun_full={rerun_full} -> processing {len(samples_to_process)} sample(s): {samples_to_process}")

Get the minimum pixel coordinates for the top left pixel corner and the maximum pixel coordinates for the bottom right corner of the desired frame/region to crop. 

In [ ]:
def get_crop_coords(dir_base_output: Path):
    # Get crop cordinates for spatial transcirptomics area from the "tissue_positions.parquet" file
    col = "pxl_col_in_fullres"
    row = "pxl_row_in_fullres" 

    tissuepos = pd.read_parquet(dir_base_output / "spatial/tissue_positions.parquet")

    min_coord = [tissuepos.loc[tissuepos[col] > 0, col].min(), tissuepos.loc[tissuepos[row] > 0, row].min()]
    max_coord = [tissuepos.loc[tissuepos[col] > 0, col].max(), tissuepos.loc[tissuepos[row] > 0, row].max()]
    print(f"Minimum coordinates: {min_coord} (col, row)")
    print(f"Maximum coordinates: {max_coord} (col, row)")

    xmin, ymin = math.floor(min_coord[0]), math.floor(min_coord[1])
    xmax, ymax = math.ceil(max_coord[0]), math.ceil(max_coord[1])
    return xmin, ymin, xmax, ymax

1. Load the image partially (full image can be way too large)
2. Crop out desired region
3. Optionally save to .tif file
4. Convert to numpy array

In [ ]:
import pyvips

def load_cropped_wsi(dir_wsi: Path, xmin, ymin, xmax, ymax) -> Image.Image:
    # Open the image (does not load it fully)
    image = pyvips.Image.new_from_file(str(dir_wsi), access="sequential")

    cropped = image.crop(xmin, ymin, xmax - xmin, ymax - ymin)

    # Convert to NumPy array
    numpy_array = np.ndarray(
        buffer=cropped.write_to_memory(),
        dtype=np.uint8,
        shape=[cropped.height, cropped.width, cropped.bands],
    )

    # Convert to PIL image
    return Image.fromarray(numpy_array)

The following cell loops over all selected samples. For each one, it deletes old patches and metadata, crops the tissue area from the WSI and cuts it into 224×224 patches. The patch positions are then saved to the sample's metadata CSV and to the combined CSV.

In [ ]:
def replace_sample_metadata(df: pd.DataFrame, target_path: Path, sample: str):
    # replaces all rows of this sample, keeps rows of other samples
    target_path.parent.mkdir(parents=True, exist_ok=True)

    if target_path.exists():
        existing_df = pd.read_csv(target_path)
        existing_df = existing_df[existing_df["Slide_ID"].astype(str) != sample]
        combined_df = pd.concat([existing_df, df], ignore_index=True)
    else:
        combined_df = df

    combined_df.to_csv(target_path, index=False)
    print(f"Metadata saved/updated at: {target_path}")


failed_samples = []

for sample in samples_to_process:
    print(f"\n================ Processing Sample: {sample} ================")
    try:
        dir_sample = basedir / sample
        dir_wsi = find_wsi(dir_sample)
        dir_base_output = dir_sample / f"output/binned_outputs/square_{bin_size}um"
        dir_patches = get_dir_patches(dir_sample)
        dir_metadata = get_dir_metadata(dir_sample)
        dir_csv_metadata = dir_metadata / f_metadata_csv

        # remove old patches + patch metadata of this sample (embedding files stay untouched)
        if dir_patches.is_dir():
            for old_patch in dir_patches.glob("*.png"):
                old_patch.unlink()
        dir_csv_metadata.unlink(missing_ok=True)
        dir_patches.mkdir(parents=True, exist_ok=True)
        dir_metadata.mkdir(parents=True, exist_ok=True)

        xmin, ymin, xmax, ymax = get_crop_coords(dir_base_output)
        wsi_image = load_cropped_wsi(dir_wsi, xmin, ymin, xmax, ymax)

        wsi_width = xmax - xmin
        wsi_height = ymax - ymin
        print(f"Processing {sample} ({dir_wsi.name}) - Size: {wsi_width}x{wsi_height}")

        all_patch_metadata = []
        saved_patches = 0
        for x in range(0, wsi_width - patch_size[0], stride):
            for y in range(0, wsi_height - patch_size[1], stride):
                patch = wsi_image.crop((x, y, x + patch_size[0], y + patch_size[1]))

                f_patch = f"patch_{saved_patches+1}.png"
                dir_patch = dir_patches / f_patch
                patch.save(dir_patch)

                # absolute position in the full WSI
                abs_x = x + xmin
                abs_y = y + ymin
                all_patch_metadata.append([sample, f_patch, abs_x, abs_y, dir_patch])
                saved_patches += 1

        print(f"Saved {saved_patches} patches for {sample}")

        metadata_df = pd.DataFrame(all_patch_metadata, columns=['Slide_ID', 'Patch_ID', 'X', 'Y', 'Path'])

        # sample directory + combined directory
        replace_sample_metadata(metadata_df, dir_csv_metadata, sample)
        replace_sample_metadata(metadata_df, dir_combined / f_metadata_csv, sample)

        del wsi_image
        gc.collect()

    except Exception as e:
        print(f"Error processing {sample}: {e}")
        failed_samples.append(sample)

print("\nProcessed samples:", [s for s in samples_to_process if s not in failed_samples])
print("Failed samples:", failed_samples)

# Extracting Patch and Token Feature Embeddings Using the UNI Model

In this section, we pass the patches created above through the pretrained UNI model. The same logic is available as a standalone script (`01_extract_embeddings.py`).

- **Input**: the patch images in `<sample>/01_preprocessing/unimodel/patches/` and their positions (`X`, `Y`) from the combined `WSI_patch_metadata.csv`. 
    - Set `only_sample` to process a single sample, or `None` to process all folders except `skip_samples`.
- **Embeddings**: for each patch, the model returns one pooled **patch embedding** and the embeddings of its 16×16 **tokens** (14×14 pixels each). Token positions are derived from the patch position and the token's row/column.
- **Output**: per sample, `WSI_patch_metadata.h5ad` (patch embeddings) and `WSI_patch_token_embeddings.h5ad` (token embeddings) are saved in `<sample>/01_preprocessing/unimodel/metadata/`. The patch embeddings are also added to `WSI_patch_metadata.h5ad` in `combined/`, replacing old rows of re-processed samples. Token embeddings are only stored per sample.

Both AnnData objects store the embeddings in `.X`, the Slide_ID, Patch_ID and position in `.obs`, and the [X, Y] coordinates in `.obsm["spatial"]`.


In [ ]:
# basedir, skip_samples and f_metadata_* are defined in the settings cell above
only_sample = None  # single Slide_ID to process, None = all samples except skip_samples
batch_size = 16
num_workers = 12

f_token_h5ad = "WSI_patch_token_embeddings.h5ad"

# master csv with patch positions (written during patch creation)
master_csv_path = dir_combined / f_metadata_csv

def check_for_patches(basedir: Path, skip_samples, only_sample=None):
    # single sample or all non-skipped samples
    if only_sample is not None:
        sample_dirs = [
            p for p in basedir.iterdir() if p.is_dir() and p.name == only_sample
        ]
        if not sample_dirs:
            raise FileNotFoundError(
                f"Sample '{only_sample}' not found as a directory under {basedir}"
            )
    else:
        sample_dirs = [
            p
            for p in basedir.iterdir()
            if p.is_dir() and p.name != "combined" and p.name not in skip_samples
        ]

    valid_patch_dirs = []
    for sample_dir in sample_dirs:
        patch_dir = get_dir_patches(sample_dir)
        if patch_dir.is_dir() and any(patch_dir.glob("*.png")):
            valid_patch_dirs.append(patch_dir)
        else:
            print(f"Skipping {sample_dir.name}: No patch PNGs found at {patch_dir}")

    if not valid_patch_dirs:
        raise FileNotFoundError(
            f"No valid sample patch directories found inside {basedir}"
        )

    print(f"Found {len(valid_patch_dirs)} valid sample patch directories.")
    return valid_patch_dirs

class PatchDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, "*.png")))
        self.transform = transform
        self.patch_ids = [os.path.basename(img) for img in self.image_paths]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, -1  # dummy label

@torch.no_grad()
def extract_patch_and_token_features(model, dataloader, device, num_prefix_tokens):
    all_patch_embeddings = []
    all_token_embeddings = []

    for batch, _ in dataloader:
        batch = batch.to(device)
        feats = model.forward_features(batch)

        # pooled embedding per patch
        patch_emb = model.forward_head(feats, pre_logits=True)
        all_patch_embeddings.append(patch_emb.cpu())

        # drop cls/register tokens, keep spatial tokens (fp16 to save memory)
        patch_tokens = feats[:, num_prefix_tokens:, :].to(torch.float16)
        all_token_embeddings.append(patch_tokens.cpu())

    patch_feats_tensor = torch.cat(all_patch_embeddings, dim=0)
    token_feats_tensor = torch.cat(all_token_embeddings, dim=0)

    return patch_feats_tensor, token_feats_tensor

In [ ]:
# cls + register tokens come first, then the token grid
num_prefix_tokens = getattr(model, "num_prefix_tokens", 1 + timm_kwargs.get("reg_tokens", 0))
grid_h, grid_w = model.patch_embed.grid_size  # (16, 16)
n_tokens = grid_h * grid_w
token_patch_px = timm_kwargs["patch_size"]    # 14

patch_directories = check_for_patches(basedir, skip_samples, only_sample=only_sample)

if not master_csv_path.exists():
    raise FileNotFoundError(f"Master metadata CSV file not found at {master_csv_path}")

print(f"Loading master spatial metadata from {master_csv_path}...")
master_metadata_df = pd.read_csv(master_csv_path, dtype={"Slide_ID": str, "Patch_ID": str, "X": float, "Y": float})

combined_patch_h5ad_path = dir_combined / f_metadata_h5ad

all_patch_adata_list = []
processed_sample_names = []

for dir_path in patch_directories:
    sample_name = dir_path.parents[2].name
    print(f"\n================ Processing Sample: {sample_name} ================")

    metadata_dir = dir_path.parent / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)

    # patch positions of current sample only
    sample_metadata_df = master_metadata_df[master_metadata_df["Slide_ID"] == sample_name]
    if sample_metadata_df.empty:
        print(f"Skipping {sample_name}: No matching metadata found in master CSV.")
        continue
    sample_positions = sample_metadata_df[["Patch_ID", "X", "Y"]]

    test_dataset = PatchDataset(root_dir=dir_path, transform=transform)
    test_dataloader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,  # keep patch order
        num_workers=num_workers,
    )

    patch_feats, token_feats = extract_patch_and_token_features(
        model, test_dataloader, device, num_prefix_tokens
    )

    # patch-level adata, X/Y attached per patch
    obs_patch = pd.DataFrame({"Slide_ID": sample_name, "Patch_ID": test_dataset.patch_ids})
    obs_patch = obs_patch.merge(sample_positions, on="Patch_ID", how="left")
    if len(obs_patch) != len(patch_feats):
        print(f"Warning: Row count mismatch in patch merge ({len(patch_feats)} -> {len(obs_patch)})!")
    obs_patch.index = sample_name + "_" + obs_patch["Patch_ID"]

    adata_patch = ad.AnnData(X=patch_feats.numpy(), obs=obs_patch)
    adata_patch.obsm["spatial"] = obs_patch[["X", "Y"]].values

    patch_h5ad_path = metadata_dir / f_metadata_h5ad
    adata_patch.write(patch_h5ad_path)
    print(f" Saved Patch AnnData: {patch_h5ad_path}")
    all_patch_adata_list.append(adata_patch)

    # token-level adata (one row per token)
    token_matrix = token_feats.reshape(-1, token_feats.shape[-1]).numpy()

    # token index within patch -> grid row/col
    token_ids = np.tile(np.arange(n_tokens), len(test_dataset.patch_ids))
    obs_tokens = pd.DataFrame({
        "Slide_ID": sample_name,
        "Patch_ID": np.repeat(test_dataset.patch_ids, n_tokens),
        "Token_ID": token_ids.astype(np.uint16),
        "Token_row": (token_ids // grid_w).astype(np.uint8),
        "Token_col": (token_ids % grid_w).astype(np.uint8),
    })
    obs_tokens = obs_tokens.merge(sample_positions, on="Patch_ID", how="left")

    # token position = patch origin + grid offset in px
    obs_tokens["Token_X"] = obs_tokens["X"] + obs_tokens["Token_col"] * token_patch_px
    obs_tokens["Token_Y"] = obs_tokens["Y"] + obs_tokens["Token_row"] * token_patch_px
    obs_tokens.index = sample_name + "_" + obs_tokens["Patch_ID"] + "_token" + obs_tokens["Token_ID"].astype(str)

    adata_token = ad.AnnData(X=token_matrix, obs=obs_tokens)
    adata_token.obsm["spatial"] = obs_tokens[["Token_X", "Token_Y"]].values

    # tokens are only stored per sample (no combined file)
    token_h5ad_path = metadata_dir / f_token_h5ad
    adata_token.write(token_h5ad_path, compression="gzip")
    print(f" Saved Sample Token AnnData: {token_h5ad_path}")

    processed_sample_names.append(sample_name)

    # free memory
    del patch_feats, token_feats, token_matrix, obs_tokens, adata_token
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# combined patch file: append new samples, replace re-processed ones
if all_patch_adata_list:
    print("\nWriting master combined Patch AnnData...")
    new_patch_adata = ad.concat(all_patch_adata_list, join="outer", index_unique=None)
    if combined_patch_h5ad_path.exists():
        existing_patch_adata = ad.read_h5ad(combined_patch_h5ad_path)
        stale_mask = existing_patch_adata.obs["Slide_ID"].astype(str).isin(processed_sample_names)
        if stale_mask.any():
            print(f"Dropping {stale_mask.sum()} pre-existing rows for re-processed sample(s) "
                  f"{sorted(set(processed_sample_names))} before appending.")
            existing_patch_adata = existing_patch_adata[~stale_mask].copy()
        new_patch_adata = ad.concat([existing_patch_adata, new_patch_adata], join="outer", index_unique=None)
    new_patch_adata.write(combined_patch_h5ad_path)

print("\nPipeline complete!")